![Foundations of NLP] Embeddings()

---



### About:  
Computers are only able to work with numerical data. One of the most important tasks for working with language is convert the text data into a numerical form. Text embeddings aim to preserve the meaning and context of words by assigning a vector to words where similar words with similar meanings or contexts would have similar vectors. In this notebook we will explore progressively more complex text embeddings and talk about how they are used. 


### Objectives:
1. Describe word2vec and cosine similarity 
1. Apply word2vec to customer reviews
1. Evaluate word2vec output
1. Describe the problem contextual embeddings aim to address 
1. Apply BERT model to a dataset and explore outputs of contextual embeddings  


### Prerequisites:
tktk

### Imports
- [gensim](https://radimrehurek.com/gensim/intro.html#installation)
- [pyTorch](https://pytorch.org/get-started/locally/)



In [2]:
#imports 
import numpy as np
import pandas as pd

## Gensisim implementation of word2vec imports 
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.models import  Word2Vec
import re # regular expression library 

# Required NLTK data downloads 
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

## Pytorch imports for BERT embeddings
import torch
from transformers import BertTokenizer, BertModel
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/amyroberts/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/amyroberts/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/amyroberts/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/Users/amyroberts/opt/anaconda3/lib/python3.8/site-packages/transformers/utils/generic.py:260: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(


### NLP Scenario 
You are an analyst for a marketing company that just launched a new product suite of mobile devices. You have data from product reviews of the new product. We need to draft a report for our client that reports the sentiment of the reviews. We will start by trying to predict the review rating using the text data as features. 


#####  Product Reviews 
1. "I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!"
2. "I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow."
3. "The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine."
4. "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it."
5. "The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it."
6.  "The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."

In [3]:
# data setup 
# create a list of reviews 
reviews = ["I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!",
"I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow.",
"The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine.",
"I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it.",
"The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it.",
"The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."
]

In [4]:
# Create a function to preprocess the text
def preprocess_text(text):
    """
    Clean and tokenize text
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    
    return tokens

# Preprocess the reviews
processed_reviews = [preprocess_text(review) for review in reviews]

In [5]:
#lets review our tokenized reviews
for review in processed_reviews:
    print(review)


['absolutely', 'love', 'techwave', 'x', 'made', 'daily', 'tasks', 'much', 'easier', 'efficient', 'highly', 'recommend']
['im', 'impressed', 'techwave', 'x', 'lacks', 'essential', 'features', 'quite', 'slow']
['techwave', 'x', 'fantastic', 'exceeded', 'expectations', 'become', 'essential', 'part', 'daily', 'routine']
['found', 'techwave', 'x', 'quite', 'average', 'job', 'theres', 'nothing', 'particularly', 'special']
['techwave', 'x', 'terrible', 'full', 'glitches', 'crashes', 'frequently', 'regret', 'purchasing']
['techwave', 'x', 'disappointing', 'doesnt', 'live', 'hype', 'missing', 'several', 'key', 'functionalities']


Compare the tokenized reviews with the original reviews. Note that the text has been converted to lowercase, special characters and numbers have been removed, and stopwords have been removed.

#####  Original Product Reviews 
1. "I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!"
2. "I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow."
3. "The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine."
4. "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it."
5. "The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it."
6.  "The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."  

### Word2Vec

Text embeddings aim to preserve the meaning and context of words by assigning a vector to words where similar words with similar meanings or contexts would have similar vectors.

<img src="./assets/food-vector-example.png" alt="Word2Vec Example" width="500" height="300">


Notice that angles between words can be used to see how similar two items are to each other. Ice cream and sorbet are very similar to each other, while salad and coffee are very different from one another. In mathematical terms, we can measure these differences using a cosine similarity.   

Cosine similarity ranges in values from -1 to 1. Where 1 is an angle (θ) that is close to 0° or perfectly aligned (ice cream and sorbet are close in our example). In contrast a value of -1 is 180° or opposite each other (coffee and salad in our example). 

Let's code this for our scenario. 


In [6]:
#initialize the word2vec model on our reviews
model = Word2Vec(sentences=processed_reviews, vector_size=100, window=5, sg=0, min_count=1)

In [7]:
# Get the word vector for a word (impressed)
model.wv['impressed']

array([-1.9496806e-03, -5.2737282e-03,  9.4501702e-03, -9.2979521e-03,
        4.4959732e-03,  5.3932490e-03, -1.4010590e-03,  9.0252776e-03,
        9.8833656e-03, -5.4926067e-03, -6.0221297e-03, -6.7429733e-03,
       -7.9084327e-03, -3.0402502e-03, -5.5951988e-03, -8.3429860e-03,
        7.9188228e-04,  2.9938079e-03,  6.4169774e-03, -2.6358613e-03,
       -4.4544656e-03,  1.2584123e-03,  4.0801967e-04,  8.1272982e-03,
        1.9533036e-04,  7.2402377e-03, -8.2719177e-03,  8.4378710e-03,
       -1.9038265e-03,  8.7142801e-03, -7.6144733e-03,  1.7934642e-03,
        1.0701027e-03,  2.3821796e-05, -5.1120762e-03, -9.2380671e-03,
       -7.2709210e-03, -7.9609165e-03,  1.9074611e-03,  4.7256550e-04,
       -1.8240417e-03,  7.1225869e-03, -2.4811502e-03, -1.3374415e-03,
       -8.8853678e-03, -9.9322619e-03,  8.9487229e-03, -5.7562315e-03,
       -6.3699363e-03,  5.2019907e-03,  6.6678785e-03, -6.8267616e-03,
        9.5003191e-04, -6.0201329e-03,  1.6458780e-03, -4.2931517e-03,
      

In [8]:
model.wv['terrible']

array([-0.00777639, -0.00675763, -0.00315397,  0.0066152 , -0.00083423,
        0.00888416, -0.00229178, -0.00530431,  0.00391531,  0.00221647,
       -0.00028038, -0.00245288, -0.00600254,  0.00403767, -0.00761157,
       -0.00890654, -0.00916153,  0.00838807,  0.00400058,  0.00709217,
        0.0098901 , -0.00722584,  0.00402851,  0.00282114,  0.00615478,
       -0.00338795,  0.00933061, -0.00579942,  0.00703221,  0.00694974,
       -0.00039196,  0.00505998,  0.00664465,  0.00154354,  0.00736852,
        0.00240867, -0.00152077,  0.0073944 ,  0.00468735, -0.00317224,
        0.00164611, -0.00013999,  0.00496326, -0.00541243, -0.00418461,
       -0.00093795,  0.00382188, -0.00658155,  0.00784563, -0.00308784,
        0.00573998,  0.00286404,  0.00682248, -0.00446627, -0.00320205,
        0.00305724, -0.00277482,  0.00594252, -0.00595423,  0.00259165,
       -0.00072277, -0.00061427,  0.00253851, -0.0021853 , -0.00425752,
       -0.0085526 ,  0.00248828,  0.00532471, -0.00586721,  0.00

In [9]:
# find the most similar words to a word
model.wv.most_similar('impressed')

[('full', 0.24841894209384918),
 ('purchasing', 0.23798175156116486),
 ('essential', 0.15064062178134918),
 ('love', 0.1492442786693573),
 ('features', 0.12813645601272583),
 ('particularly', 0.1216324120759964),
 ('job', 0.11494724452495575),
 ('part', 0.11391659826040268),
 ('missing', 0.09712193161249161),
 ('techwave', 0.09433545172214508)]

In [10]:
model.wv.most_similar('terrible')

[('essential', 0.19554859399795532),
 ('several', 0.12082455307245255),
 ('hype', 0.09498394280672073),
 ('x', 0.0949057787656784),
 ('disappointing', 0.09437797218561172),
 ('love', 0.08960356563329697),
 ('regret', 0.08144096285104752),
 ('frequently', 0.07116393744945526),
 ('highly', 0.06835755705833435),
 ('efficient', 0.054231975227594376)]

Notice that the word 'impressed' is similar to 'purchasing' and 'love', while the word 'terrible' is similar to 'hype' and 'disappointing'. 

This shows that the Word2Vec model has learned the some of the semantic relationships between words based on the context in which they appear in the reviews. Given our very small training set this is rather impressive. With more data and more examples of usage these would improve. 

### Understanding word order 

Let’s look at the following example:

- "The supplier agreed to pay the manufacturer"
- "The manufacturer agreed to pay the supplier"



#### Try it!  
Use the steps above to convert these to vectors using word to vec 

In [11]:
# create a list of sentences
sentences = ["The supplier agreed to pay the manufacturer",
             "The manufacturer agreed to pay the supplier"]


# Preprocess the sentences using the function we created earlier, preprocess_text()
processed_reviews = [preprocess_text(sentence) for sentence in sentences]

# Initialize the Word2Vec model
model = Word2Vec(sentences=processed_reviews, vector_size=100, window=5, sg=0, min_count=1)


In [12]:
# Get the word vector for a word (supplier)
model.wv['supplier']

array([-8.2426779e-03,  9.2993546e-03, -1.9766092e-04, -1.9672764e-03,
        4.6036304e-03, -4.0953159e-03,  2.7431143e-03,  6.9399667e-03,
        6.0654259e-03, -7.5107943e-03,  9.3823504e-03,  4.6718083e-03,
        3.9661205e-03, -6.2435055e-03,  8.4599797e-03, -2.1501649e-03,
        8.8251876e-03, -5.3620026e-03, -8.1294188e-03,  6.8245591e-03,
        1.6711927e-03, -2.1985089e-03,  9.5136007e-03,  9.4938548e-03,
       -9.7740470e-03,  2.5052286e-03,  6.1566923e-03,  3.8724565e-03,
        2.0227872e-03,  4.3050171e-04,  6.7363144e-04, -3.8206363e-03,
       -7.1402504e-03, -2.0888723e-03,  3.9238976e-03,  8.8186832e-03,
        9.2591504e-03, -5.9759365e-03, -9.4026709e-03,  9.7643770e-03,
        3.4297847e-03,  5.1661171e-03,  6.2823449e-03, -2.8042626e-03,
        7.3227035e-03,  2.8302716e-03,  2.8710044e-03, -2.3803699e-03,
       -3.1282497e-03, -2.3701417e-03,  4.2764368e-03,  7.6057913e-05,
       -9.5842788e-03, -9.6655441e-03, -6.1481940e-03, -1.2856961e-04,
      

In [13]:
# Get the word vector for a word (manufacturer)
model.wv['agreed']

array([ 9.4563962e-05,  3.0773198e-03, -6.8126451e-03, -1.3754654e-03,
        7.6685809e-03,  7.3464094e-03, -3.6732971e-03,  2.6427018e-03,
       -8.3171297e-03,  6.2054861e-03, -4.6373224e-03, -3.1641065e-03,
        9.3113566e-03,  8.7338570e-04,  7.4907029e-03, -6.0740625e-03,
        5.1605068e-03,  9.9228229e-03, -8.4573915e-03, -5.1356913e-03,
       -7.0648370e-03, -4.8626517e-03, -3.7785638e-03, -8.5361991e-03,
        7.9556061e-03, -4.8439382e-03,  8.4236134e-03,  5.2625705e-03,
       -6.5500261e-03,  3.9578713e-03,  5.4701497e-03, -7.4265362e-03,
       -7.4057197e-03, -2.4752307e-03, -8.6257253e-03, -1.5815723e-03,
       -4.0343284e-04,  3.2996845e-03,  1.4418805e-03, -8.8142155e-04,
       -5.5940580e-03,  1.7303658e-03, -8.9737179e-04,  6.7936908e-03,
        3.9735902e-03,  4.5294715e-03,  1.4343059e-03, -2.6998555e-03,
       -4.3668128e-03, -1.0320747e-03,  1.4370275e-03, -2.6460087e-03,
       -7.0737829e-03, -7.8053069e-03, -9.1217868e-03, -5.9351693e-03,
      

#### What did you notice? 

These two sentences would have identical representations in word2vec but have very different meanings. In this example, the order of the words is critical to the meaning of the phrase. We aren't able to capture this order with the bag-of-words approach used in word2vec and other similar models. To address this issue, methods were developed to capture the sequential order of words. 


### Contextual Embeddings 

Unlike word2vec, contextual embeddings create different embeddings for the same word depending on the context in which it was used. To capture the context we use information such as the words around our word of interest, its sentence position, and other factors to map its vector. In other words, contextual embeddings are position aware. 

Let's look at our example from before: 
- "The supplier agreed to pay the manufacturer"
- "The manufacturer agreed to pay the supplier"

We will use BERT, a famous language model that uses transformer encoder architecture, to create contextual embeddings.  
We can access this model using [pyTorch transformers](https://pytorch.org/hub/huggingface_pytorch-transformers/) . 



In [14]:
# Example sentences
sentence1 = "The supplier agreed to pay the manufacturer"
sentence2 = "The manufacturer agreed to pay the supplier"

In [15]:
# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

The first step is to create our tokens 

In [16]:
# Add special tokens and convert to tensor 
input1 = tokenizer(sentence1, return_tensors="pt", padding=True, truncation=True)

# Take a look at the tokenized sentence
tokenizer.convert_ids_to_tokens(input1['input_ids'][0])

['[CLS]', 'the', 'supplier', 'agreed', 'to', 'pay', 'the', 'manufacturer', '[SEP]']


Notice that the BERT tokenizer added two special tokens `[CLS]` is BERT's way of marking the start of the input document (a sentence in this case). `[SEP]` indicates a separation (such as between sentences) and the end of a document. 

### Try it! 
Create the tokens for sentence 2

In [17]:
# Add special tokens and convert to tensor 
input2 = tokenizer(sentence2, return_tensors="pt", padding=True, truncation=True)

# Take a look at the tokenized sentence
tokenizer.convert_ids_to_tokens(input2['input_ids'][0])

['[CLS]',
 'the',
 'manufacturer',
 'agreed',
 'to',
 'pay',
 'the',
 'supplier',
 '[SEP]']

Let's create a few functions to create the tokens and then use them to create the BERT embeddings at the sentence level and the token level. For now, focus on the outputs. We will dive into how to the details of how code block below works at another time. 

In [18]:
# Sentence level embeddings
def get_bert_embedding(sentence):
    """
    Get BERT embeddings for a sentence
    """
    # Add special tokens and convert to tensor
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
    
    # Get BERT embeddings
    with torch.no_grad(): # No need to calculate gradients. This will save memory and time 
        outputs = model(**inputs)
        
    # Get the embeddings from the last hidden state
    last_hidden_states = outputs.last_hidden_state
    
    # Return both the sentence-level embedding (CLS token) and token-level embeddings
    return {
        'sentence_embedding': last_hidden_states[0][0].numpy(),  # CLS token
        'token_embeddings': last_hidden_states[0].numpy(),      # All tokens
        'tokens': tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    }

Now we can pass each of our sentences to the function to get the embeddings for each. 

In [19]:
#print each sentence as a reminder of what we are working with
print(sentence1)
print(sentence2)

The supplier agreed to pay the manufacturer
The manufacturer agreed to pay the supplier


In [20]:
# Get BERT embeddings for the sentences
embedding1 = get_bert_embedding(sentence1)
embedding2 = get_bert_embedding(sentence2)

In [21]:
# Calculate the cosine similarity between the sentence embeddings
similarity = cosine_similarity([embedding1['sentence_embedding']], [embedding2['sentence_embedding']])[0][0]
print(f"Sentence Level Cosine Similarity: {similarity}")

Sentence Level Cosine Similarity: 0.996477484703064


Notice that our two sentences are very similar. This is expected because they contain the exact same words! But let's look at the embeddings for the individual tokens (words in this case). 

Interesting they are no longer perfectly similar. Let's look at the individual embeddings to compare the word 'supplier' in both sentences. 

In [22]:
#Let's compare the embeddings for supplier in both sentences
#sentence 1
id1 = embedding1['tokens'].index('supplier')
print("Sentence 1")
print(embedding1['token_embeddings'][id1][:10]) # print only first 10 values of the embedding
print("")

#sentence 2
id2 = embedding2['tokens'].index('supplier')
print("Sentence 2")
print(embedding2['token_embeddings'][id2][:10]) # print only first 10 values of the embedding

Sentence 1
[ 0.9074207  -0.43041566  0.32430553  0.1308197   0.04717512  0.23647887
 -0.2693361   0.41109467  1.2377744   0.02060751]

Sentence 2
[ 0.19103675 -0.46897602  0.00414992  0.13056713 -0.34808046  0.44466808
 -0.34096095  0.27727354  1.0207361  -0.07885481]


Same word but different embeddings! That's because the order in which they were used (sequence) was used in the embedding. Below we can see the cosine similarity for both versions of supplier. 

In [23]:
supplier_similarity = cosine_similarity(
            embedding1['token_embeddings'][id1].reshape(1, -1),
            embedding2['token_embeddings'][id2].reshape(1, -1)
        )[0][0]
print(f"Similarity between 'supplier' in both sentences: {supplier_similarity}")

Similarity between 'supplier' in both sentences: 0.7415687441825867


### Try it! 
Checkout the embeddings for another word of your choice. 

In [24]:
#Let's compare the embeddings for 'pay' in both sentences
#sentence 1
id1 = embedding1['tokens'].index('pay')
print("Sentence 1")
print(embedding1['token_embeddings'][id1][:10]) # print only first 10 values of the embedding
print("")

#sentence 2
id2 = embedding2['tokens'].index('pay')
print("Sentence 2")
print(embedding2['token_embeddings'][id2][:10]) # print only first 10 values of the embedding
print("")

pay_similarity = cosine_similarity(
            embedding1['token_embeddings'][id1].reshape(1, -1),
            embedding2['token_embeddings'][id2].reshape(1, -1)
        )[0][0]
print(f"Similarity between 'pay' in both sentences: {pay_similarity}")

Sentence 1
[ 1.75651863e-01 -1.36638544e-02  3.45966190e-01  2.97459602e-01
  1.11937486e-01  4.56450313e-01 -6.91465288e-02 -9.44722354e-01
  5.65952301e-01 -9.48831439e-06]

Sentence 2
[ 0.17775811 -0.1218427   0.3169069   0.31182224  0.03333663  0.4593305
 -0.15798827 -0.73284173  0.46566436  0.11598112]

Similarity between 'pay' in both sentences: 0.9909030795097351


Notice that "pay" has a very high similarity across the two sentences. 

### Conclusion
In this notebook, we learned how to use word embeddings to represent text data. We started by using the Word2Vec model to generate word embeddings for a list of reviews. This method worked really well but wasn't able to capture the impact of sequence of words on meaning. Because word2vec is a bag-of-words method, a given word will have the same embedding regardless of where it is used in a sentence. 

We then used the pre-trained BERT model to generate contextual embeddings for a pair of sentences. With this method, we were able to create different embeddings for the same word depending on how it was used in a sentence. These contextual embeddings are a big leap forward and are foundational for common AI uses, such as ChatGPT. 

### Recommended readings 
#### Primary sources 
[Foundational word2vec paper](https://arxiv.org/abs/1301.3781)
[Foundational BERT paper](https://arxiv.org/pdf/1810.04805) 

#### Other helpful references 
tktk

### 